添加一个新的预估命令，这个命令就是能在原有的基数估计上实现，带推理谓词的基数估计，先只考虑树采样，对于满足结构匹配的结果，再经过一次条件检查:只有对应post节点的oracle列满足>0.5，才会success++。对于post的oracle调用的结果已经保存到post.csv文件中，可以预先缓存，通过hash查找的方式直接查找满足子图匹配嵌入中推理节点的oracle结果。


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import re
import sys
import math
import time
import tempfile
import traceback
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
import json
from pythonProject.src.Structure_first.fastest_pipeline import FastestGraphConverter, FastestEstimateMerger
from pythonProject.src.Structure_first.graph_sample import FastestRunner
from pythonProject.src.Structure_first.precision_submatching import ExactSubgraphMatcher
from pythonProject.src.Structure_first.proxy_sample import ProxyStratifiedSampler, compute_T_true

# 一级测试数据集
datasets_name = "parler_data"
# 一级数据集下根据查询和图结构的差异划分的子测试数据集
dataset_name = "dataset_three"
# dataset_name = "dataset_one"
# 原始CSV数据路径
CSV_BASE_DIR = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/csv_data"
# 转换后GraphLib数据存放路径
Graph_Lib_Dir = f"/home/wangshuo/resource/datasets/{datasets_name}/{dataset_name}/data_graph"

# converter = FastestGraphConverter(CSV_BASE_DIR,Graph_Lib_Dir)
# converter.run_without_author_user_post()
# converter.remove_edge_labels()

In [ ]:
# 初始化 Runner
runner = FastestRunner(build_dir="/home/wangshuo/projects/FaSTest-main/build")

dataset_name = "dataset_three"
current_budget = 20000
infer_label = 1  # 对应 Post 节点的标签

# 调用 run 方法
code, output = runner.run(
    dataset=dataset_name,
    root_label=infer_label,           # 必须指定推理节点的标签
    sample_budget=current_budget,     # 设置预算
    estimate_with_predicate=True      # <--- 开启单推理谓词模式
)